# 05 Modelo Híbrido

Combinación de señales colaborativas y de contenido.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from surprise import Dataset, Reader, SVD
from pathlib import Path
base = Path(r'd:\GitHub\Ruta_aprendizaje_2024\03-aplicación_de_modelos\proyecto_3\data')
ratings = pd.read_csv(base / 'ratings.csv')
movies = pd.read_csv(base / 'movies.csv')
tags = pd.read_csv(base / 'tags.csv')

In [ ]:
reader = Reader(rating_scale=(ratings['rating'].min(), ratings['rating'].max()))
data = Dataset.load_from_df(ratings[['userId','movieId','rating']], reader)
trainset = data.build_full_trainset()
svd = SVD(random_state=42)
svd.fit(trainset)

In [ ]:
tags_agg = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(map(str,x)))
movies['genres_text'] = movies['genres'].str.replace('|',' ')
movies = movies.merge(tags_agg, left_on='movieId', right_index=True, how='left')
movies['tag'] = movies['tag'].fillna('')
corpus = (movies['genres_text'].fillna('') + ' ' + movies['tag']).values
tfidf = TfidfVectorizer(min_df=2)
X = tfidf.fit_transform(corpus)
sim = cosine_similarity(X)

## Fusión de puntajes

In [ ]:
user_id = ratings['userId'].iloc[0]
rated = set(ratings.loc[ratings['userId']==user_id,'movieId'])
candidates = movies.loc[~movies['movieId'].isin(rated)].copy()
collab_scores = candidates['movieId'].apply(lambda m: svd.predict(user_id, int(m)).est)
title = movies['title'].iloc[0]
idx = movies.index[movies['title']==title][0]
content_scores = pd.Series(sim[idx][candidates.index], index=candidates.index)
collab_norm = (collab_scores - collab_scores.min())/(collab_scores.max()-collab_scores.min())
content_norm = (content_scores - content_scores.min())/(content_scores.max()-content_scores.min())
alpha = 0.6
hybrid = alpha*collab_norm + (1-alpha)*content_norm
candidates.loc[:, 'score'] = hybrid.values
candidates.sort_values('score', ascending=False).head(10)[['movieId','title','score']]